# PoC Reset Notebook

This notebook clears PoC run data from Fabric tables so the pipeline can be executed from a clean slate.

Safety guardrails:
- Destructive operations run only when `confirmation_token` is exactly `RESET_POC_DATA`.
- Uses `TRUNCATE TABLE` to preserve schemas expected by setup notebooks.
- Leaves `/Files/config/*` intact by default.

In [ ]:
# Parameters (set these before running)
confirmation_token = "NO"  # Must be exactly RESET_POC_DATA
include_stage1_raw_tables = True
include_ai_tables = True
drop_reconciliation_objects = True

try:
    confirmation_token = mssparkutils.notebook.params.get("confirmation_token") or confirmation_token
except Exception:
    pass

print(f"confirmation_token={confirmation_token}")
print(f"include_stage1_raw_tables={include_stage1_raw_tables}")
print(f"include_ai_tables={include_ai_tables}")
print(f"drop_reconciliation_objects={drop_reconciliation_objects}")

In [ ]:
# Safety gate
if confirmation_token != "RESET_POC_DATA":
    raise ValueError(
        "Refusing to run reset. Set confirmation_token to RESET_POC_DATA to proceed."
    )

print("Safety gate passed. Proceeding with reset.")

In [ ]:
# Build candidate object lists
core_tables = [
    "source_dfm_raw",
    "individual_dfm_consolidated",
    "aggregated_dfms_consolidated",
    "policy_aggregates",
    "tpir_load_equivalent",
    "dq_results",
    "dq_exception_rows",
    "validation_events",
    "parse_errors",
    "schema_drift_events",
    "run_audit_log",
    "tpir_check_result",
    "ads_load_audit",
    "run_summary"
]

stage1_raw_tables = [
    "stage1_brown_shipley_positions_raw",
    "stage1_brown_shipley_cash_raw",
    "stage1_castlebay_raw",
    "stage1_wh_ireland_raw",
    "stage1_pershing_positions_raw",
    "stage1_pershing_psl_valuation_raw",
    "stage1_pershing_other_raw"
]

ai_tables = [
    "ai_triage_labels",
    "ai_resolution_suggestions",
    "ai_narrative_summary",
    "ai_anomaly_events"
]

tables_to_truncate = list(core_tables)
if include_stage1_raw_tables:
    tables_to_truncate.extend(stage1_raw_tables)
if include_ai_tables:
    tables_to_truncate.extend(ai_tables)

print("Tables selected for truncate:")
for t in tables_to_truncate:
    print(f" - {t}")

In [ ]:
# Preview current row counts
for t in tables_to_truncate:
    try:
        if spark.catalog.tableExists(t):
            cnt = spark.table(t).count()
            print(f"{t}: {cnt} rows")
        else:
            print(f"{t}: <not found>")
    except Exception as e:
        print(f"{t}: <error reading count> {e}")

In [ ]:
# Execute reset
for t in tables_to_truncate:
    if spark.catalog.tableExists(t):
        spark.sql(f"TRUNCATE TABLE {t}")
        print(f"TRUNCATED {t}")
    else:
        print(f"SKIPPED {t} (not found)")

if drop_reconciliation_objects:
    spark.sql("DROP VIEW IF EXISTS vw_holdings_price_validation")
    spark.sql("DROP TABLE IF EXISTS holdings_price_validation")
    print("Dropped reconciliation objects if they existed")

In [ ]:
# Verify zero-row state
all_clear = True
for t in tables_to_truncate:
    if spark.catalog.tableExists(t):
        cnt = spark.table(t).count()
        print(f"{t}: {cnt} rows after reset")
        if cnt != 0:
            all_clear = False

print("\nReset complete")
print(f"all_clear={all_clear}")